In [ ]:
import heapq
import math

class State:
  
  def __init__(self, row, col, grid):
    self.row = row
    self.col = col
    self.grid = grid
    self.n = len(grid)


    #checking if the current cell is (n-1)(n-1)th cell
  def goalTest(self):
    return self.row == self.n - 1 and self.col == self.n - 1

  def moveGen(self):
    directions = [(-1, -1), (-1, 0), (-1, 1),
                  (0, -1),          (0, 1),
                  (1, -1),  (1, 0), (1, 1)]
    children = []
    for dr, dc in directions:
      new_row, new_col = self.row + dr, self.col + dc
      # Check if the new position is within the grid boundaries
      if 0 <= new_row < self.n and 0 <= new_col < self.n:
        # Check if the cell is part of a clear path (value is 0)  
        if self.grid[new_row][new_col] == 0:
          children.append(State(new_row, new_col, self.grid))
    return children
 
    
  #helper functions

  def __eq__(self, other):
    return self.row == other.row and self.col == other.col

  def __hash__(self):
    return hash((self.row, self.col))

  def __lt__(self, other):
    return (self.row, self.col) < (other.row, other.col)

  def __repr__(self):
    return f"({self.row}, {self.col})"
  

#implementation of Best first search and A* algorithm
class Search:
  def __init__(self, start_state):
    self.start_state = start_state
    self.goal_state = State(start_state.n - 1, start_state.n - 1, start_state.grid)
    # Check for invalid start/goal points
    if start_state.grid[start_state.row][start_state.col] == 1 or \
       start_state.grid[self.goal_state.row][self.goal_state.col] == 1:
        self.is_solvable = False
    else:
        self.is_solvable = True

    #function to calculate heuristic value
  def _heuristic(self, state):
    return math.sqrt((state.row - self.goal_state.row)**2 + (state.col - self.goal_state.col)**2)

  def _reconstructPath(self, closed):
    path = []
    parent_map = {node: parent for node, parent in closed}
    curr = self.goal_state
    while curr is not None:
      path.append(curr)
      curr = parent_map.get(curr)
    return path[::-1] 
  
    #impementation of best first search
  def bestFirstSearch(self):
    if not self.is_solvable:
        return None

    # open is a priority queue: (h_cost, (node, parent))
    open_q = [(self._heuristic(self.start_state), (self.start_state, None))]
    # closed is a set for efficient lookup of visited nodes
    closed_set = set()
    # closed_list stores (node, parent) pairs for path reconstruction
    closed_list = []

    while open_q:
      h_cost, (node, parent) = heapq.heappop(open_q)

      if node in closed_set:
        continue

      closed_set.add(node)
      closed_list.append((node, parent))

      if node.goalTest():
        return self._reconstructPath(closed_list)

      for child in node.moveGen():
        if child not in closed_set:
          heapq.heappush(open_q, (self._heuristic(child), (child, node)))

    return None  

  def aStarSearch(self):
    if not self.is_solvable:
        return None

    # open is a priority queue: (f_cost, g_cost, (node, parent))
    # g_cost is the path length from the start node  
    open_q = [(self._heuristic(self.start_state) + 1, 1, (self.start_state, None))]
    # closed_set stores nodes already processed to avoid cycles
    closed_set = set()
    # closed_list stores (node, parent) pairs for path reconstruction
    closed_list = []

    while open_q:
      f_cost, g_cost, (node, parent) = heapq.heappop(open_q)

      if node in closed_set:
        continue

      closed_set.add(node)
      closed_list.append((node, parent))

      if node.goalTest():
        return self._reconstructPath(closed_list)

      for child in node.moveGen():
        if child not in closed_set:
          new_g_cost = g_cost + 1
          h_cost = self._heuristic(child)
          f_cost = new_g_cost + h_cost
          heapq.heappush(open_q, (f_cost, new_g_cost, (child, node)))
          
    return None  

def run_and_print(grid):
    start_state = State(0, 0, grid)
    search_problem = Search(start_state)

    print("Best First Search")
    path_bfs = search_problem.bestFirstSearch()
    if path_bfs:
        print(f"Path length: {len(path_bfs)}, Path: {[repr(node) for node in path_bfs]}")
    else:
        print("Path length: -1")

    print("\nA* Search")
    path_astar = search_problem.aStarSearch()
    if path_astar:
        print(f"Path length: {len(path_astar)}, Path: {[repr(node) for node in path_astar]}")
    else:
        print("Path length: -1")
 
print("Example 1")
grid1 = [[0, 1], [1, 0]] 
run_and_print(grid1)
print("-" * 25)

print("\n Example 2")
grid2 = [[0, 0, 0], [1, 1, 0], [1, 1, 0]] 
run_and_print(grid2)
print("-" * 25)

print("\n  Example 3  ")
grid3 = [[1, 0, 0], [1, 1, 0], [1, 1, 0]]  
run_and_print(grid3)
print("-" * 25)

Example 1
Best First Search
Path length: 2, Path: ['(0, 0)', '(1, 1)']

A* Search
Path length: 2, Path: ['(0, 0)', '(1, 1)']
-------------------------

 Example 2
Best First Search
Path length: 4, Path: ['(0, 0)', '(0, 1)', '(1, 2)', '(2, 2)']

A* Search
Path length: 4, Path: ['(0, 0)', '(0, 1)', '(1, 2)', '(2, 2)']
-------------------------

  Example 3  
Best First Search
Path length: -1

A* Search
Path length: -1
-------------------------
